# Correlation


## I think I get this



In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
Correlation function object
"""
from __future__ import annotations

import json
from typing import Callable, Optional
import numpy as np

from tenso.bath.distribution import BoseEinstein
from tenso.bath.sd import SpectralDensity
from tenso.bath.aaa import aaa
from numpy.typing import NDArray

PI = np.pi


class Correlation(object):

    def __init__(self) -> None:
        self.coefficients = list()  # type: list[complex]
        self.conj_coefficents = list()  # type: list[complex]
        self.zeropoints = list()  # type: list[complex]
        self.derivatives = dict()  # type: dict[tuple[int, int], complex]
        self.lindblad_rate = None  # type: Optional[float]
        return

    def dump(self, output_file: str) -> None:
        with open(output_file, 'w') as f:
            c = [(_c.real, _c.imag) for _c in self.coefficients]
            cc = [(_cc.real, _cc.imag) for _cc in self.conj_coefficents]
            z = [(_z.real, _z.imag) for _z in self.zeropoints]
            d = {
                f"{i},{j}": (_d.real, _d.imag)
                for (i, j), _d in self.derivatives.items()
            }
            kwargs = {
                'coefficients': c,
                'conj_coefficents': cc,
                'zeropoints': z,
                'derivatives': d,
                'lindblad_rate': self.lindblad_rate,
            }
            json.dump(kwargs, f, indent=4, sort_keys=True)
        return

    def remove_heom_terms(self) -> None:
        self.coefficients = list()
        self.conj_coefficents = list()
        self.zeropoints = list()
        self.derivatives = dict()
        return

    def load(self, input_file: str) -> None:
        with open(input_file, 'r') as f:
            kwargs = json.load(f)
            c = [complex(x, y) for x, y in kwargs['coefficients']]
            cc = [complex(x, y) for x, y in kwargs['conj_coefficents']]
            z = [complex(x, y) for x, y in kwargs['zeropoints']]
            dct = kwargs['derivatives']  # type: dict[str, tuple[float, float]]
            d = dict()
            for string, (x, y) in dct.items():
                idx = string.split(',')
                i = int(idx[0])
                j = int(idx[1])
                d[i, j] = complex(x, y)
            lr = kwargs['lindblad_rate']
            assert len(c) == len(cc) == len(z)
            self.coefficients = c
            self.conj_coefficents = cc
            self.zeropoints = z
            self.derivatives = d
            self.lindblad_rate = lr
        return

    @property
    def k_max(self):
        assert len(self.coefficients) == len(self.coefficients) == len(
            self.zeropoints)
        return len(self.coefficients)

    def add_discrete_vibration(self, frequency: float, coupling: float,
                               beta: Optional[float]) -> None:
        w0 = frequency
        g = coupling

        coth = 1.0 / np.tanh(beta * w0 / 2.0) if beta is not None else 1.0
        self.coefficients.extend(
            [g**2 / 2.0 * (coth + 1.0), g**2 / 2.0 * (coth - 1.0)])
        self.conj_coefficents.extend(
            [g**2 / 2.0 * (coth - 1.0), g**2 / 2.0 * (coth + 1.0)])
        self.zeropoints.extend([1.0, 1.0])
        k = len(self.derivatives)
        self.derivatives[k, k] = -1.0j * w0
        self.derivatives[k + 1, k + 1] = 1.0j * w0
        return

    def add_discrete_trigonometric(self, frequency: float, coupling: float,
                                   beta: Optional[float]) -> None:
        w0 = frequency
        g = coupling

        coth = 1.0 / np.tanh(beta * w0 / 2.0) if beta is not None else 1.0
        c1 = g**2 / 2.0 * (coth + 1.0)
        c2 = g**2 / 2.0 * (coth - 1.0)
        self.coefficients.extend([(c2 + c1), (c2 - c1) * 1.0j])
        self.conj_coefficents.extend([(c2 + c1).conj(),
                                      ((c2 - c1) * 1.0j).conj()])
        self.zeropoints.extend([1.0, 0.0])  # cos * exp, sin * exp
        k = len(self.derivatives)
        self.derivatives[k, k + 1] = -w0
        self.derivatives[k + 1, k] = w0
        return

    def _add_ltc(self, sds: list[SpectralDensity], distribution: BoseEinstein):
        """Add LTC terms for spectral densities with poles.
        """
        residue_pairs = distribution.residues
        if sds and residue_pairs:
            for res, pole in residue_pairs:
                cs = [-2.0j * PI * res * sd.function(pole) for sd in sds]
                c = np.sum(cs)
                self.coefficients.append(c)
                self.conj_coefficents.append(np.conj(c))
                self.zeropoints.append(1.0)
                k = len(self.derivatives)
                self.derivatives[k, k] = -1.0j * pole

        return

    def add_spectral_densities(self,
                               sds: list[SpectralDensity],
                               distribution: BoseEinstein,
                               zeropoint=1.0,
                               use_ht_function=False):
        f = distribution.function if not use_ht_function else distribution.ht_function
        for sd in sds:
            rs, ps = sd.get_residues_poles()
            if len(rs) == 1:
                c = rs[0] * f(ps[0])
                self.coefficients.append(c / zeropoint)
                self.conj_coefficents.append(c.conjugate() / zeropoint)
                self.zeropoints.append(zeropoint)
                k = len(self.derivatives)
                self.derivatives[k, k] = -1.0j * ps[0]
            elif len(rs) == 2:
                c1 = rs[0] * f(ps[0])
                c2 = rs[1] * f(ps[1])
                self.coefficients.extend([c1 / zeropoint, c2 / zeropoint])
                self.conj_coefficents.extend(
                    [c2.conjugate() / zeropoint,
                     c1.conjugate() / zeropoint])
                self.zeropoints.extend([zeropoint, zeropoint])
                k = len(self.derivatives)
                self.derivatives[k, k] = -1.0j * ps[0]
                self.derivatives[k + 1, k + 1] = -1.0j * ps[1]
            else:
                raise RuntimeError(
                    'Poles must be symmetric along the imag axis.')

        self._add_ltc(sds, distribution)
        return

    def add_trigonometric(self, sds: list[SpectralDensity],
                          distribution: BoseEinstein):
        f = distribution.function
        for sd in sds:
            rs, ps = sd.get_residues_poles()
            if len(rs) == 2:
                # ps = [-1.0j * (g + 1.0j * w), -1.0j * (g - 1.0j * w)]
                g = (ps[0] + ps[1]) * 0.5j
                w = (ps[0] - ps[1]) * 0.5
                c1 = rs[0] * f(ps[0])  # for term exp[(- iw - g) t]
                c2 = rs[1] * f(ps[1])  # for term exp[(+ iw - g) t]
                self.coefficients.extend([(c2 + c1), (c2 - c1) * 1.0j])
                self.conj_coefficents.extend([(c2 + c1).conj(),
                                              ((c2 - c1) * 1.0j).conj()])
                self.zeropoints.extend([1.0, 0.0])  # cos * exp, sin * exp
                k = len(self.derivatives)
                self.derivatives[k, k] = -g
                self.derivatives[k, k + 1] = -w
                self.derivatives[k + 1, k] = w
                self.derivatives[k + 1, k + 1] = -g
            elif len(rs) == 1:
                c = rs[0] * f(ps[0])
                self.coefficients.append(c)
                self.conj_coefficents.append(c.conj())
                self.zeropoints.append(1.0)
                k = len(self.derivatives)
                self.derivatives[k, k] = -1.0j * ps[0]
            else:
                raise RuntimeError(
                    'Poles must be symmetric along the imag axis.')

        self._add_ltc(sds, distribution)
        return

    def real_correlation_function(self, t):
        ans = np.zeros_like(t)
        for k, c in enumerate(self.coefficients):
            g = complex(self.derivatives[k, k])
            ans += c.real * np.exp(g.real * t) * np.cos(g.imag * t)
            ans -= c.imag * np.exp(g.real * t) * np.sin(g.imag * t)
        return ans

    def imag_correlation_function(self, t):
        ans = np.zeros_like(t)
        for k, c in enumerate(self.coefficients):
            g = complex(self.derivatives[k, k])
            ans += c.real * np.exp(g.real * t) * np.sin(g.imag * t)
            ans += c.imag * np.exp(g.real * t) * np.cos(g.imag * t)
        return ans

    def __str__(self) -> None:
        if self.k_max > 0:
            string = f"Correlation ( c | c* | z ) x{self.k_max} :"
            for c, cc, z in zip(self.coefficients, self.conj_coefficents,
                                self.zeropoints):
                string += f"\n{c.real:+.4e}{c.imag:+.4e}j | {cc.real:+.4e}{cc.imag:+.4e}j | {z.real:+.2e}{z.imag:+.2e}j"
            string += "\nDerivatives:"
            string += "".join([
                f"\n    [{i:d}, {j:d}] : {v.real:+.4e}{v.imag:+.4e}j"
                for (i, j), v in self.derivatives.items()
            ])
        else:
            string = 'No HEOM correlations.'
        if self.lindblad_rate is not None:
            string += f'\nLindblad rate: {self.lindblad_rate:.4e}'
        else:
            string += '\nNo Lindblad rate.'
        return string


def get_corr_from_aaa(spfs: list[Callable[[NDArray], NDArray]],
                      freq_space,
                      beta,
                      dual=False,
                      tol=1e-13,
                      k_max=100):
    """
    Get the correlation function from the spectral functions.
    """
    corr = Correlation()
    be = BoseEinstein(n=0, beta=beta).function
    if not dual:
        freq_space = freq_space
    else:
        freq_space = np.concatenate((-freq_space[::-1], freq_space), axis=None)
    jw = np.array(sum(spf(freq_space) for spf in spfs))
    jbw = jw * be(freq_space)
    # import matplotlib.pyplot as plt
    # plt.plot(dual_freq_space, jw)
    # plt.plot(dual_freq_space, jbw)
    # plt.show()
    # plt.close()

    res = aaa(jbw, freq_space, tol=tol, mmax=k_max, return_errors=False)
    poles, residues = res.polres()
    mask = np.imag(poles) < 0
    poles = poles[mask]
    residues = -2.0j * np.pi * residues[mask]

    k_max = len(poles)
    corr.coefficients = [r for r in residues] + [0.0 for _ in residues]
    corr.conj_coefficents = [0.0 for _ in residues] + \
        [r.conjugate() for r in residues]
    corr.zeropoints = [1.0] * 2 * k_max
    corr.derivatives = {(k, k): -1.0j * p for k, p in enumerate(poles)}
    corr.derivatives.update({
        (k + k_max, k + k_max): (-1.0j * p).conjugate()
        for k, p in enumerate(poles)
    })
    return corr


"""
class RealCorrelation(Correlation):
    # Real correlation function that contains only real basis functions.
    def add_spectral_densities(self, sds: list[SpectralDensity],
                               distribution: BoseEinstein):
        f = distribution.function

        for sd in sds:
            k = len(self.derivatives)
            rs, ps = sd.get_residues_poles()
            if len(rs) == 1:
                c = rs[0] * f(ps[0])
                g = (-1.0j * ps[0])
                self.coefficients.extend([c.real, 1.0j * c.imag])
                self.conj_coefficents.extend([c.real, -1.0j * c.imag])
                self.zeropoints.extend([1.0, 1.0])
                self.derivatives[k, k + 1] = g
                self.derivatives[k + 1, k] = g
            elif len(rs) == 2:
                g1 = -1.0j * ps[0]
                g2 = -1.0j * ps[1]
                c1 = rs[0] * f(ps[0])  # for term exp[(- iw - g) t]
                c2 = rs[1] * f(ps[1])  # for term exp[(+ iw - g) t]
                self.coefficients.extend(
                    [c1.real, 1.0j * c1.imag, c2.real, 1.0j * c2.imag])
                self.conj_coefficents.extend(
                    [c2.real, -1.0j * c2.imag, c1.real, -1.0j * c1.imag])
                self.zeropoints.extend([1.0, 1.0, 0.0,
                                        0.0])  # cos * exp, sin * exp
                self.derivatives[k, k + 1] = g1
                self.derivatives[k + 1, k] = g1
                self.derivatives[k + 2, k + 3] = g2
                self.derivatives[k + 3, k + 2] = g2
            else:
                raise NotImplementedError

        self._add_ltc(sds, distribution)
        return
"""

if __name__ == "__main__":
    # test aaom a
    import matplotlib.pyplot as plt
    from tenso.libs.quantity import Quantity as __
    from tenso.bath.sd import OhmicExp, Drude
    unit = 1000  # cm-1
    freq_space = np.linspace(0, 10, 1000)[1:]
    # freq_space = np.logspace(np.log10(1e-6), np.log10(3), 1000, base=10)
    beta = __(1 / 300, '/K').au * __(unit, '/cm').au
    sd = Drude(0.2, 0.1)
    # sd = OhmicExp(0.2, 0.1)
    print(f'{type(sd).__name__} @ {beta}')
    corr = get_corr_from_aaa([sd.function], freq_space, beta, 1e-8, 100)
    print(corr)
    sd2 = OhmicExp(0.2, 0.1)
    print(f'{type(sd2).__name__} @ {beta}')
    freq_space = np.linspace(0, 1, 1000)[1:]
    corr2 = get_corr_from_aaa([sd2.function], freq_space, beta, 1e-3, 100)
    print(corr2)

    # Plot the spectral density
    plt.plot(freq_space,
             sd.function(freq_space).real,
             'k-',
             lw=1,
             label=f'{type(sd).__name__}')
    plt.plot(freq_space,
             sd2.function(freq_space).real,
             'b-',
             lw=1,
             label=f'{type(sd2).__name__}')
    plt.legend()
    plt.xlabel('Frequency')
    plt.title('Spectral Density')
    plt.show()
    plt.close()

    # Plot the poles
    data = [corr.derivatives[k, k] for k in range(corr.k_max)]
    data = np.array(data) * 1.0j
    plt.plot(data.real, data.imag, 'o', label=f'{type(sd).__name__}')
    data2 = [corr2.derivatives[k, k] for k in range(corr2.k_max)]
    data2 = np.array(data2) * 1.0j
    plt.plot(data2.real, data2.imag, 'x', label=f'{type(sd2).__name__}')
    plt.legend()
    plt.title('Poles')
    # plt.plot(data[km//2:].real, data[km//2:].imag, 'x')
    plt.show()
    plt.close()

    # Plot the residues
    km = corr.k_max
    data = [corr.coefficients[k] for k in range(km)]
    idx = np.arange(km)
    data = np.array(data)
    plt.plot(idx, data[:km].real, 'ko', label=f'Re {type(sd).__name__}')
    plt.plot(idx, data[:km].imag, 'bo', label=f'Im {type(sd).__name__}')
    km = corr2.k_max
    data2 = [corr2.coefficients[k] for k in range(km)]
    idx = np.arange(km)
    data2 = np.array(data2)
    plt.plot(idx, data2[:km].real, 'kx', label=f'Re {type(sd2).__name__}')
    plt.plot(idx, data2[:km].imag, 'bx', label=f'Im {type(sd2).__name__}')
    plt.title('Residues')
    plt.legend()
    # plt.plot(data[km//2:].real, data[km//2:].imag, 'x')
    plt.show()
    plt.close()

    # Plot the function
    t = np.linspace(0, 100, 1000)
    plt.plot(t, corr.real_correlation_function(t), 'k-', lw=1, label='Real')
    plt.plot(t, corr.imag_correlation_function(t), 'b-', lw=1, label='Imag')
    plt.plot(t, corr2.real_correlation_function(t), 'k:', lw=2)
    plt.plot(t, corr2.imag_correlation_function(t), 'b:', lw=2)

    # plt.plot(t, corr2.real_correlation_function(t), 'k:', lw=2)
    # plt.plot(t, corr2.imag_correlation_function(t), 'b:', lw=2)

    # Plot the reference
    corr_ref = Correlation()
    corr_ref.add_spectral_densities([Drude(0.2, 0.1)],
                                    BoseEinstein(n=5, beta=beta))
    print(corr_ref)
    plt.plot(t,
             corr_ref.real_correlation_function(t),
             'k-.',
             lw=3,
             label='Re (Pade)')
    plt.plot(t,
             corr_ref.imag_correlation_function(t),
             'b-.',
             lw=3,
             label='Im (Pade)')
    plt.legend()
    plt.show()